In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import s3fs

MINIO_ENDPOINT = "http://169.58.96.176:9000"
MINIO_KEY = "admin"
MINIO_SECRET = "adminGrupo3"
BUCKET_NAME = "programacion-ciencia-datos-g3"

storage_opts = {
    "key": MINIO_KEY,
    "secret": MINIO_SECRET,
    "client_kwargs": {"endpoint_url": MINIO_ENDPOINT}
}

print("🔄 1. Ejecutando integración (5.4)...")
df_trans = pd.read_parquet(f"s3://{BUCKET_NAME}/silver/train_transaction_transformed.parquet", storage_options=storage_opts)
df_id = pd.read_parquet(f"s3://{BUCKET_NAME}/silver/train_identity_transformed.parquet", storage_options=storage_opts)
df_master = pd.merge(df_trans, df_id, on="TransactionID", how="left")
print(f"✨ Tabla maestra integrada: {df_master.shape[0]:,} filas y {df_master.shape[1]} columnas.")

print("\n🛠️ 2. Iniciando preprocesamiento de datos (5.5)...")
df_ml = df_master.copy()

# ==============================================================================
# 5.5 PREPROCESAMIENTO DE DATOS
# ==============================================================================

# 1._ Detección de outliers (si aplica)
# Aplicamos transformación logarítmica (log1p) para suavizar valores extremos y asimetría en montos
df_ml['log_TransactionAmt'] = np.log1p(df_ml['TransactionAmt'])

# 2._ Normalización (si aplica)
# Estandarizamos las variables numéricas clave para unificar escalas (Media = 0, Desviación Estándar = 1)
scaler = StandardScaler()
cols_a_escalar = ['log_TransactionAmt', 'monto_pen', 'C1', 'D1']
for col in cols_a_escalar:
    df_ml[col] = df_ml[col].fillna(df_ml[col].median())
df_ml[cols_a_escalar] = scaler.fit_transform(df_ml[cols_a_escalar])

# 3._ Encoding (si aplica)
# Convertimos las variables categóricas del negocio en matrices numéricas binarias mediante One-Hot Encoding
cols_categoricas_ml = ['card4', 'card6', 'ProductCD', 'franja_horaria', 'DeviceType', 'familia_os']
df_encoded = pd.get_dummies(df_ml, columns=cols_categoricas_ml, drop_first=True, dtype=int)

# 4._ Preparación final del dataset
# Separamos la matriz de características predictoras (X) de la variable objetivo (y = isFraud)
columnas_excluir = ['TransactionID', 'isFraud', 'fecha_hora', 'fecha_dia', 'dia_semana', 'mes_operacion', 'id_30', 'id_31', 'DeviceInfo', 'P_emaildomain']
features = [c for c in df_encoded.columns if c not in columnas_excluir]

X = df_encoded[features]
y = df_encoded['isFraud']

print(f"✨ ¡Dataset preprocesado con éxito y listo para el modelado!")
print(f"   • Matriz de Features (X): {X.shape[0]:,} registros y {X.shape[1]} variables.")
print(f"   • Vector Objetivo (y):   {y.shape[0]:,} registros (Tasa de Fraude: {y.mean()*100:.2f}%)")
display(X.head(3))

🔄 1. Ejecutando integración (5.4)...
✨ Tabla maestra integrada: 590,540 filas y 31 columnas.

🛠️ 2. Iniciando preprocesamiento de datos (5.5)...
✨ ¡Dataset preprocesado con éxito y listo para el modelado!
   • Matriz de Features (X): 590,540 registros y 36 variables.
   • Vector Objetivo (y):   590,540 registros (Tasa de Fraude: 3.50%)


,TransactionDT,TransactionAmt,card1,card2,addr1,dist1,C1,D1,hora_dia,monto_pen,...,franja_horaria_Mañana (06-11h),franja_horaria_Noche (18-23h),franja_horaria_Tarde (12-17h),DeviceType_desktop,DeviceType_mobile,familia_os_Linux,familia_os_MacOS,familia_os_Otro/Desconocido,familia_os_Windows,familia_os_iOS
0,86400,68.5,13926.0,387.0,315.0,19.0,-0.09802,-0.508897,0,-0.278161,...,0,0,0,0,0,0,0,0,0,0
1,86401,29.0,2755.0,404.0,325.0,9.0,-0.09802,-0.597765,0,-0.443327,...,0,0,0,0,0,0,0,0,0,0
2,86469,59.0,4663.0,490.0,330.0,287.0,-0.09802,-0.597765,0,-0.317889,...,0,0,0,0,0,0,0,0,0,0
